# CredResolve – Is the "11% month-on-month recovery improvement" real?
Run top-to-bottom from the `Analysis/` folder. The notebook calls the same scripts a reviewer can run from the repository root (`python src/clean.py` etc.), so every number here is reproducible. Raw files in `collections_30k_dataset/` are never modified. All headline numbers live in `reports/headline_numbers.json`.

**Evidence labels used:** Fact · Strong evidence · Correlation · Hypothesis. **All statistical tests here are exploratory** (one synthetic extract, 212 daily values): "not significant" means *no statistically detectable* effect, not proof of no effect.

In [1]:
import os, subprocess, pandas as pd
os.chdir('..') if os.path.basename(os.getcwd()) == 'Analysis' else None   # run everything from repo root
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)
def run(script):
    r = subprocess.run(['python', '-W', 'ignore', f'src/{script}'], capture_output=True, text=True); print(r.stdout[-3000:] if r.returncode == 0 else r.stderr)

## 1. Profile the raw data
17 CSVs. Coverage is **1 Jan – 8 Aug 2026 (≈7.2 months)**, not 12. Row counts, key uniqueness and duplicates:

In [2]:
import glob
rows = []
keys = {'borrowers':'borrower_id','accounts':'account_id','agents':'agent_id','calls':'call_id','payments':'payment_id','promises_to_pay':'ptp_id','call_dispositions':'disposition_id','whatsapp_events':'whatsapp_event_id'}
for t, k in keys.items():
    d = pd.read_csv(f'collections_30k_dataset/{t}.csv', low_memory=False)
    rows.append({'table': t, 'rows': len(d), 'unique_keys': d[k].nunique(), 'repeated_keys': int(d[k].duplicated().sum()), 'exact_duplicate_rows': int(d.duplicated().sum()), 'null_cells': int(d.isna().sum().sum())})
pd.DataFrame(rows)

,table,rows,unique_keys,repeated_keys,exact_duplicate_rows,null_cells
0,borrowers,30600,11015,19585,600,1509
1,accounts,30000,30000,0,0,455
2,agents,30000,1000,29000,0,0
3,calls,91350,90000,1350,1271,1827
4,payments,25500,25000,500,486,382
5,promises_to_pay,18000,18000,0,0,0
6,call_dispositions,35000,35000,0,0,0
7,whatsapp_events,60600,60000,600,600,0


## 2. Golden dataset (`src/clean.py`)
Rules: exact duplicates dropped; same-ID conflicts merged (null filled, first-ingested row kept, other version logged); borrowers/agents **not** resolved (all rows kept and labelled UNRESOLVED because `updated_at < created_at` in 15,063 borrower rows and one agent_id has ~29 employee codes); timestamps converted only for calls (they carry a timezone label) and only as an *assumption*.

In [3]:
run('clean.py')

        calls                removed: later versions of conflicting IDs    79                                                                              
          calls              corrected: null fields filled from duplicate    68                                                                              
          calls                    flagged: agent_id still missing (kept)  1759                                                                              
          calls flagged: outside 1-Jan..8-Aug-2026 window (kept, flagged)     5                                                                              
          calls                                               golden rows 90000                                                                              
promises_to_pay                                                  raw rows 18000                                                                              
promises_to_pay                                       

In [4]:
pd.read_csv('reports/cleaning_summary.csv')

,table,step,rows,note
0,accounts,raw rows,30000,NaN
1,accounts,flagged: borrower_id missing (kept),455,not imputed: borrower_id in other tables does ...
2,accounts,flagged: borrower_id not found in borrowers (k...,2458,NaN
3,accounts,golden rows,30000,NaN
4,payments,raw rows,25500,NaN
5,payments,removed: exact duplicate rows,486,NaN
6,payments,"conflicting payment_id (same ID, different con...",14,kept first-ingested row; null fields filled fr...
7,payments,removed: later versions of conflicting IDs,14,NaN
8,payments,corrected: null fields filled from duplicate,14,NaN
9,payments,flagged: SUCCESS payments > outstanding amount...,1862,NaN


## 3. Metrics (`src/metrics.py`) – definitions in `reports/metric_definitions.md`

In [5]:
run('metrics.py')

.74            2.32
KEPT          4180           4.09            2.61
OPEN          4128           3.61            2.47
OPEN promises already past promised_date (stale status): 4128 of 16758
Unmatured PTPs excluded: 1242
# Statistical tests (zero-filled calendar, 1 Jan - 31 Jul 2026)

- Calendar days: 212 | days with zero SUCCESS recovery (filled as 0): 0 | days present in raw data: 212
- Feb: 28 days, mean ₹60.77 lakh/day | Mar: 31 days, mean ₹60.94 lakh/day
- Total Feb→Mar change: +11.03% | Per-day change: +0.29%
- Welch t-test (daily means): t=0.09, p=0.932
- Mann-Whitney U: p=0.885
- Bootstrap (10,000 resamples of days) 95% CI for per-day change: [-5.9%, +7.1%]
- Permutation test (shuffle day labels Feb/Mar): p=0.935
- Linear trend, Jan–Jul daily recovery: slope ₹-697/day per day (-0.35% of mean per 30 days), p=0.456
- Kendall tau trend test: tau=-0.040, p=0.384
- Day-of-week index (1.00 = average; Mon..Sun): 1.01, 0.98, 1.03, 1.01, 1.00, 0.97, 1.00
- Month-over-month swings in tot

In [6]:
pd.read_csv('reports/monthly_metrics.csv', index_col='month').T.round(2)

month,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08
days_in_data,31.00,28.00,31.00,30.00,31.00,30.00,31.00,8.00
recovery_cr,18.72,17.01,18.89,17.51,18.42,17.56,18.72,4.71
recovery_cr_per_day,0.60,0.61,0.61,0.58,0.59,0.58,0.60,0.59
n_success_payments,2464.00,2268.00,2524.00,2406.00,2449.00,2366.00,2441.00,616.00
accounts_paying,2374.00,2173.00,2419.00,2304.00,2344.00,2286.00,2335.00,608.00
pct_accounts_paying,7.91,7.24,8.06,7.68,7.81,7.62,7.78,2.03
payments_per_paying_account,1.04,1.04,1.04,1.04,1.04,1.03,1.04,1.01
avg_ticket_rs,75985.85,75018.72,74846.42,72792.20,75234.90,74201.07,76707.20,76476.78
recovery_per_account_rs,6240.97,5671.42,6297.08,5837.94,6141.68,5851.99,6241.41,1570.32
recovery_rate_pct_of_outstanding,1.78,1.62,1.80,1.67,1.76,1.67,1.78,0.45


## 4. The 11% claim
**Fact (arithmetic):** the only +11% pair is Feb→Mar total SUCCESS recovery. February has 28 days, March 31. Decomposition: calendar days × paying accounts per day × payments per account × ticket. The report below also shows serial-correlation diagnostics, a Newey–West (HAC) trend and a 7-day block bootstrap, because daily values may be correlated.

In [7]:
print(open('reports/statistical_tests.md').read())

# Statistical tests (zero-filled calendar, 1 Jan - 31 Jul 2026)

- Calendar days: 212 | days with zero SUCCESS recovery (filled as 0): 0 | days present in raw data: 212
- Feb: 28 days, mean ₹60.77 lakh/day | Mar: 31 days, mean ₹60.94 lakh/day
- Total Feb→Mar change: +11.03% | Per-day change: +0.29%
- Welch t-test (daily means): t=0.09, p=0.932
- Mann-Whitney U: p=0.885
- Bootstrap (10,000 resamples of days) 95% CI for per-day change: [-5.9%, +7.1%]
- Permutation test (shuffle day labels Feb/Mar): p=0.935
- Linear trend, Jan–Jul daily recovery: slope ₹-697/day per day (-0.35% of mean per 30 days), p=0.456
- Kendall tau trend test: tau=-0.040, p=0.384
- Day-of-week index (1.00 = average; Mon..Sun): 1.01, 0.98, 1.03, 1.01, 1.00, 0.97, 1.00
- Month-over-month swings in total recovery (Feb..Jul): -9.1%, +11.0%, -7.3%, +5.2%, -4.7%, +6.7% | all explained by days: -9.7%, +10.7%, -3.2%, +3.3%, -3.2%, +3.3% (calendar-day ratio)

## Feb→Mar decomposition (total recovery ratio = product of factor

**Reading:** per-day recovery moved +0.3% (block-bootstrap 95% CI −5.2% to +5.9%; naive CI −5.9% to +7.1%), so the data do not establish a statistically significant improvement in daily recovery and a per-day gain of 11% is not supported. **No statistically detectable trend was identified in the observed period** (HAC slope −0.35% per 30 days, 95% CI −1.05% to +0.35%, p=0.327). This is not proof that performance is flat: changes of a few percent cannot be ruled out. Composition of payments did not shift detectably (chi-square p>0.05), and mix-standardised ticket moved −0.2/−0.3% vs raw −0.2%.

*Simpson's paradox / mix:* segment-level and overall ticket moved the same way. *Survivorship / denominator:* see section 4b.

## 4b. Recovery-rate denominators (`src/denominators.py`)
Which denominators are defensible? Recovery divided by the static outstanding balance is only an **index**: 75% of accounts are PAID/CLOSED/WRITEOFF yet pay at the same rate as ACTIVE ones, and the status history contradicts the accounts table.

In [ ]:
run('denominators.py')

In [ ]:
pd.read_csv('reports/denominator_by_status.csv')

## 5. Independent validation from RAW files with SQL

In [8]:
run('validate.py')

                                             sum  count
query    metric                                        
calls    accounts_worked_outbound              8      8
         contact_rate_call_pct                 8      8
         outbound_calls                        8      8
ptp      ptp_matured                           8      8
         ptp_recorded_kept_pct_of_matured      8      8
         ptp_validated_kept_any_payment_pct    8      8
recovery accounts_paying                       8      8
         avg_ticket_rs                         8      8
         n_success_payments                    8      8
         recovery_cr                           8      8
ALL MATCH



## 6. PTP validation, attribution and channel checks (`src/ptp_validation.py`, `src/investigate.py`)

In [9]:
run('investigate.py')

attribution window  1d:   1.4% of SUCCESS payments /   1.4% of rupees have a prior call
attribution window  3d:   4.1% of SUCCESS payments /   4.1% of rupees have a prior call
attribution window  7d:   9.0% of SUCCESS payments /   9.1% of rupees have a prior call
attribution window 14d:  16.6% of SUCCESS payments /  16.8% of rupees have a prior call
attribution window 30d:  31.6% of SUCCESS payments /  31.6% of rupees have a prior call
attribution window 60d:  49.9% of SUCCESS payments /  49.7% of rupees have a prior call
PLACEBO call in N days AFTER payment: {7: np.float64(9.0), 30: np.float64(31.3)}
Payments where 'latest campaign ever' differs from last-touch-before-payment (30d): 89.6 % (incl. no-touch)
SUCCESS payment accounts never called: 5.3 % | share of all accounts never called: 5.3 %
            mean   size
answered               
False     0.4394  16465
True      0.4470  13535
          mean   size
called               
False   0.4322   1592
True    0.4434  28408
{'FIELD': 

**Reading:** PTPs are restricted to *matured* promises (1,242 un-matured excluded) and validated against SUCCESS payments in [created, promised date]. A payment can fall inside several promises of one account, so the kept rate is a **range**: 3.67% (each payment credited to one promise) to 3.71% (overlap allowed); only 8 payments sit in 2+ windows. Recorded KEPT status (≈50% of KEPT+BROKEN) is not supported by payments, and a same-length placebo window after the due date gives 3.71%. Attribution to prior calls equals a placebo, and contact/payment rates are the same for every campaign channel / target status. Payments carry no campaign link, so channel ROI is not identifiable.

In [10]:
run('ptp_validation.py')

,status,matured,validated_any,validated_full
0,BROKEN,4220,3.39,2.23
1,CANCELLED,4230,3.74,2.32
2,KEPT,4180,4.09,2.61
3,OPEN,4128,3.61,2.47


## 7. Driver screen (`src/drivers.py`) – correlation only

In [11]:
run('drivers.py')

                                driver  outcome  groups  min_rate_pct  max_rate_pct  chi2_p                             verdict
                      telephony vendor answered       5         19.49         20.68   0.050           no evidence of difference
                 vendor schema_version answered       3         19.68         20.12   0.344           no evidence of difference
             campaign strategy_version answered       4         19.76         20.01   0.927           no evidence of difference
                      campaign channel answered       5         19.44         20.25   0.414           no evidence of difference
                 hour band (as stored) answered       5         19.70         20.46   0.528           no evidence of difference
                   call timezone label answered       3         19.74         20.12   0.504           no evidence of difference
                            DPD bucket answered       5         19.75         20.23   0.829           no

18 screens were run (14 contact/payment drivers + WhatsApp, SMS and field-event screens); ~0.9 false positives are expected at 5%. The one hit (DPD bucket vs paying, p=0.028) does not survive a Bonferroni threshold (0.0028) → **Correlation (weak)**. Agent, geography, tenure and language cannot be analysed reliably (identity conflicts; no language field).

## 8. Counterfactual design (targeting change) – methodology only

The data does **not** contain an identifiable targeting change (campaign `strategy_version` start dates overlap Jan–May; payments have no campaign link), so an effect cannot be estimated. Proposed design:
- **Treatment:** accounts targeted under the new strategy from date T. **Control:** comparable accounts not targeted (or targeted later: staggered roll-out).
- **Identification:** difference-in-differences on account-week payment rate, with account and week fixed effects; matching on DPD, loan type, risk, outstanding, pre-period payment history (propensity score).
- **Assumptions:** parallel pre-trends (check with event-study plot), no spillover, stable measurement. **Confounders:** seasonality/month length, portfolio ageing, vendor changes, collections-team changes. **Limitations:** needs an assignment log and payment-to-campaign linkage that do not exist here.
- **Best option:** randomise a holdout (see sizing below).

## 9. ₹10 Cr question and experiment sizing – no invented ROI (`src/investment_sizing.py`)
The sample size is computed on a **unique-account cohort** (first targeting date as index, 14-day SUCCESS payment outcome). The earlier row-based 11,557 per arm is withdrawn because `daily_targeting` repeats accounts.

In [12]:
run('investment_sizing.py')

# Investment sizing (measured facts + labelled sensitivities)

- Measured: Jan–Jul 2026 SUCCESS recovery = ₹126.8 Cr over 212 days = ₹0.598 Cr/day
- Annualised run-rate (x365, assumes no seasonality – an extrapolation): ₹218 Cr
- Break-even arithmetic: ₹10 Cr spend needs ₹10 Cr / c of incremental recovery, where c = share of recovered rupees CredResolve actually keeps (UNKNOWN – not in the data).

| assumed c (keep rate) | incremental recovery needed (₹ Cr) | needed as % of annualised run-rate |
|---:|---:|---:|
| 100% | 10 | 4.6% |
| 50% | 20 | 9.2% |
| 25% | 40 | 18.3% |
| 10% | 100 | 45.8% |

(The c values are sensitivities, NOT estimates. One-off spend, 12-month horizon; no discounting.)

## Experiment sizing

- Measured baseline: 3.59% of daily-target rows are followed by a SUCCESS payment within 14 days (same for every channel and target status → no measurable channel effect today)

| relative lift to detect | absolute lift | accounts needed PER ARM (80% power, 5% two-sided) |
|-

In [ ]:
print(open('reports/investment_recommendation.md').read()[:3000])

In [ ]:
print(open('reports/experiment_design.md').read()[:2500])

**Conclusion:** the data cannot support a reliable ROI for any of the six options (no cost or keep-rate data, no measured channel/vendor effect, no payment-campaign link). On the unique-account baseline of 3.76%, detecting a +20% relative lift needs about 11,015 accounts per arm (+50%: 1,988). Scenario values are hypothetical parameters, not estimates. Recommended action: stage the money and run a randomised holdout; see `reports/investment_recommendation.md`, `reports/experiment_design.md` and `reports/executive_memo.md`.